In [72]:
import sqlite3
import pandas as pd

import json
import sqlite3
import sys
import os
from pathlib import Path
from pprint import pprint

In [32]:
ROOT = Path(os.getcwd()).parents[0]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SCRIPTS = ROOT / "scripts"
if SCRIPTS not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

In [44]:
from load_questions import load_questions
from config import setting, setting_path
from pipeline.run import run_simulation

from ingest.sqlite import connect, connect_sqlalchemy
from pipeline.inference import run_inference
from pipeline.prepare import (
    create_instructions,
    create_state,
    fetch_personas,
    prepare_campaign_info,
    prepare_question_info,
)


# Variables

In [ ]:

SELECTED_COLUMNS = {
    "id": "persona_id",
    "persona_column": "persona_summary",
    "other_columns": ["region", "gender", "age_band"],
}

MAX_PERSONAS = 3


In [37]:
questions_db = setting_path("QUESTIONS_DB")
persona_db = setting_path("PERSONA_DB")
model_name = setting("MODEL_NAME")

simulation_db = setting_path("SIMULATION_DB")
simulation_name = setting("SIMULATION_NAME")
persona_table = setting("PERSONA_TABLE")


In [ ]:
print(f"Loading questions into {questions_db}", flush=True)
load_questions(questions_db)

# Get demo questions

In [35]:
def reaction_questions(db_path: Path) -> list[dict[str, object]]:
    """Return the score questions stored in the reactions table."""
    with sqlite3.connect(db_path) as conn:
        rows = conn.execute(
            """
            SELECT name, instructions_json, criteria_json
            FROM reactions
            ORDER BY rowid
            """
        ).fetchall()

    questions: list[dict[str, object]] = []
    for name, instructions_json, criteria_json in rows:
        instructions = json.loads(instructions_json)
        questions.append(
            {
                "question_name": name,
                "question_type": "score",
                "instructions": instructions["question"],
                "focus": instructions["focus"],
                "criteria": json.loads(criteria_json),
                "is_saved": 0,
            }
        )
    return questions


def parse_campaigns(db_path: Path) -> list[dict[str, object]]:
    """Return one campaign per loaded question row.

    A score row is an advertisement. Its questions are the shared reactions.
    A noul or choice row is one campaign with one question.
    """
    reactions = reaction_questions(db_path)
    with sqlite3.connect(db_path) as conn:
        rows = conn.execute(
            """
            SELECT category, campaign, instructions_json, criteria_json
            FROM questions
            ORDER BY rowid
            """
        ).fetchall()

    campaigns: list[dict[str, object]] = []
    next_question_id = 1
    ad_number = 0
    for category, campaign, instructions_json, criteria_json in rows:
        if category == "score":
            ad_number += 1
            questions: list[dict[str, object]] = []
            for reaction in reactions:
                question = dict(reaction)
                question["id"] = next_question_id
                next_question_id += 1
                questions.append(question)
            name = f"ad {ad_number}"
        else:
            instructions = json.loads(instructions_json)
            questions = [
                {
                    "id": next_question_id,
                    "question_name": instructions["question"],
                    "question_type": category,
                    "instructions": instructions["question"],
                    "focus": instructions["focus"],
                    "criteria": json.loads(criteria_json),
                    "is_saved": 0,
                }
            ]
            next_question_id += 1
            name = str(instructions["question"])

        campaigns.append(
            {
                "name": name,
                "description": campaign,
                "is_saved": 0,
                "questions": questions,
            }
        )
    return campaigns

In [43]:
print(f"Parsing campaigns from {questions_db}", flush=True)
campaigns = parse_campaigns(questions_db)
question_count = sum(len(campaign["questions"]) for campaign in campaigns)


yes_no_campaign = campaigns[0]
choice_campaign = campaigns[4]
score_campaign = campaigns[8]

Parsing campaigns from /Users/goku/repos/simulator/data/demo_questions.db


# Get personas

In [49]:
personas, _ = fetch_personas(
    persona_db, persona_table, [], SELECTED_COLUMNS, 1
)

In [52]:
persona = list(personas.values())[0]

# Run simulation

## yes_no campaign

In [76]:
state = create_state(persona, yes_no_campaign["description"])
instructions = create_instructions(0, yes_no_campaign["questions"])

simulated_response = run_inference(model_name, instructions, state)

In [80]:
answer = simulated_response.answers
print(answer)

{'Will you buy the Sony WH-1000XM5 headphones this week for $320, with free shipping?': NoulAnswer(type='noul', noul=0.49)}


## choice campaign

In [81]:
state = create_state(persona, choice_campaign["description"])
instructions = create_instructions(0, choice_campaign["questions"])

simulated_response = run_inference(model_name, instructions, state)

In [82]:
answer = simulated_response.answers
print(answer)

{'Which option will you use for those three dinners?': ChoiceAnswer(type='choice', choice='HelloFresh', confidence=0.94, probabilities={'HelloFresh': 0.97, 'Groceries': 0.03})}


## score campaign

In [83]:
state = create_state(persona, score_campaign["description"])
instructions = create_instructions(0, score_campaign["questions"])

simulated_response = run_inference(model_name, instructions, state)

In [87]:
answer = simulated_response.answers
pprint(answer)

{'attractive': ScoreAnswer(type='score', score=1.56, confidence=0.42, legend={0: {'what': 'You do not find this advertisement attractive.', 'signals': ['There is nothing in the advertisement that you like.', 'You would look away from it.']}, 1: {'what': 'You find this advertisement slightly attractive.', 'signals': ['One moment in the advertisement is pleasant.', 'The rest of the advertisement is forgettable.']}, 2: {'what': 'You find this advertisement attractive.', 'signals': ['You would watch the advertisement through to the end.', 'You like the scene.']}, 3: {'what': 'You find this advertisement very attractive.', 'signals': ['You like the people in the advertisement.', 'You would be glad to see the advertisement again.']}, 4: {'what': 'You find this advertisement extremely attractive.', 'signals': ['Watching the advertisement is a pleasure.', 'You would show the advertisement to someone else.']}}, probabilities={0: 0.08, 1: 0.43, 2: 0.37, 3: 0.11, 4: 0.01}),
 'distasteful': ScoreA